# Notebook 5 - MLOps: Testing & CI/CD

This notebook adds automated testing to the CFPB complaint prediction pipeline.
Rather than trusting the pipeline manually, we write tests that verify:
1. Our artifacts (model, encoders, schemas) load correctly
2. LLM responses are validated against the expected schema before being used
3. Input rows are built correctly from form data + LLM features

These tests are then wired into GitHub Actions, so they run automatically
on every push, catching regressions before they reach production.

## Step 1: Set working directory and confirm artifacts are present

In [2]:
import os
os.chdir('/content/drive/MyDrive/cfpb_project')
print(os.listdir())

['Notebook 2 - Feature Engineering.ipynb', 'Notebook 3 - Model Training.ipynb', 'Notebook 1  - Data Wrangling and EDA.ipynb', 'Notebook 4 - LLM Enrichment.ipynb', 'app.py', 'llm_categorical_cols.json', 'model_enriched_v2.json', 'target_encoder.pkl', 'enriched_cols.json', 'baseline_cols.json', 'extraction_tool.json', 'CFPB Complaint Investigation Predictor · Streamlit.html', '__pycache__', '.pytest_cache', 'test_cfpb_logic.py', 'cfpb_logic.py']


## Step 2: Load core artifacts

Before writing any logic, we define a single function to load all four
JSON artifacts used throughout the pipeline: the baseline column list,
the enriched column list (including LLM-derived features), the LLM
categorical column list, and the extraction tool schema. Centralizing
this in one function avoids repeating file-loading code, and gives us
one place to test that artifacts load correctly.

In [2]:
%%writefile cfpb_logic.py
import json
import pandas as pd

def load_artifacts():
  '''Load the core JSON artifacts used by the cfpb prediction pipeline'''
  with open('baseline_cols.json') as f:
    baseline_cols = json.load(f)
  with open('enriched_cols.json') as f:
    enriched_cols = json.load(f)
  with open('llm_categorical_cols.json') as f:
    llm_categorical_cols = json.load(f)
  with open('extraction_tool.json') as f:
    extraction_tool = json.load(f)
  return baseline_cols, enriched_cols, llm_categorical_cols, extraction_tool

Writing cfpb_logic.py


## Step 2b: Sanity check that artifacts load correctly

In [3]:
from cfpb_logic import load_artifacts

# Unapcking load_artifacts
baseline_cols, enriched_cols, llm_categorical_cols, extraction_tool = load_artifacts()

print(f"Baseline columns: {len(baseline_cols)}")
print(f"Enriched columns: {len(enriched_cols)}")
print(f"LLM categorical columns: {len(llm_categorical_cols)}")
print(f"Ectraction tool name: {extraction_tool['name']}")

Baseline columns: 8
Enriched columns: 26
LLM categorical columns: 4
Ectraction tool name: extract_complaint_features


## Step 3: Validate LLM responses against the extraction tool schema

Claude's tool-use API enforces the schema at generation time, but our
pipeline should not blindly trust any upstream response. This function
checks that a response dict contains all required fields, and that any
field with a restricted set of allowed values (enum) actually matches
one of those values. This guards against a malformed or unexpected
response silently breaking the encoding step downstream.

In [7]:
# Rewriting the cfpb_logic.py

%%writefile cfpb_logic.py
import json
import pandas as pd

def load_artifacts():
  '''Load the core JSON artifacts used by the cfpb prediction pipeline'''
  with open('baseline_cols.json') as f:
    baseline_cols = json.load(f)
  with open('enriched_cols.json') as f:
    enriched_cols = json.load(f)
  with open('llm_categorical_cols.json') as f:
    llm_categorical_cols = json.load(f)
  with open('extraction_tool.json') as f:
    extraction_tool = json.load(f)
  return baseline_cols, enriched_cols, llm_categorical_cols, extraction_tool

def validate_llm_response(response:dict, extraction_tool:dict)-> bool:
  '''Check that an LLM response has a;; the required fields with allowed enum vlaues'''
  schema = extraction_tool['input_schema']['properties']
  required = extraction_tool['input_schema']['required']

  for field in required:
    if field not in response:
      raise ValueError(f"Missing rquired field {field}")

    if 'enum' in schema[field] and response[field] not in schema[field]['enum']:
      raise ValueError(f"Invalid value for {field}: {response[field]}")
  return True

Overwriting cfpb_logic.py


In [6]:
with open('cfpb_logic.py') as f:
  print(f.read())

import json
import pandas as pd

def load_artifacts():
  '''Load the core JSON artifacts used by the cfpb prediction pipeline'''
  with open('baseline_cols.json') as f:
    baseline_cols = json.load(f)
  with open('enriched_cols.json') as f:
    enriched_cols = json.load(f)
  with open('llm_categorical_cols.json') as f:
    llm_categorical_cols = json.load(f)
  with open('extraction_tool.json') as f:
    extraction_tool = json.load(f)
  return baseline_cols, enriched_cols, llm_categorical_cols, extraction_tool

def validate_llm_response(response:dict, extraction_tool:dict)-> bool:
  '''Check that an LLM response has a;; the required fields with allowed enum vlaues'''
  schema = extraction_tool['input_schema']['properties']
  required = extraction_tool['input_schema']['required']

  for field in required:
    if field not in response:
      raise ValueError(f"Missing rquired field {field}")
    
    if 'enum' in schema[field] and response[field] not in schema[field]['enum']:
      raise

In [11]:
# Fix caching issue by restarting pyhton runtime
import importlib
import cfpb_logic
importlib.reload(cfpb_logic)
from cfpb_logic import load_artifacts, validate_llm_response

In [12]:
# Sanity check with Good response
from cfpb_logic import validate_llm_response

good_response = {
    'harm_type': 'financial_loss',
    'severity': 'high',
    'discrimination_mentioned': False,
    'resolution_requested': 'refund',
    'sentiment_intensity': 'irate'
}

print(validate_llm_response(good_response, extraction_tool))

True


In [13]:
# Sanity check with a Bad response
bad_response = {
     'harm_type': 'made_up_value',
    'severity': 'high',
    'discrimination_mentioned': False,
    'resolution_requested': 'refund',
    'sentiment_intensity': 'irate'
}

try:
  validate_llm_response(bad_response, extraction_tool)
except ValueError as e:
  print('Correctly caught:',e)

Correctly caught: Invalid value for harm_type: made_up_value


## Sanity check
Good response → passes and returns True. Bad response (invalid enum) → correctly raises a ValueError.

## Step 4: Write pytest tests for artifact loading and response validation

We now write formal tests covering:
1. Artifacts load with the correct structure and expected counts
2. A valid LLM response passes validation
3. A response with an invalid enum value is correctly rejected
4. A response missing a required field is correctly rejected

Running these via pytest gives a repeatable, automated check that this
logic keeps working correctly as the project evolves.

In [21]:
# pytest
%%writefile test_cfpb_logic.py
import pytest
from cfpb_logic import load_artifacts, validate_llm_response

def test_load_artifacts_returns_correct_counts():
  baseline_cols, enriched_cols, llm_categorical_cols, extraction_tool = load_artifacts()
  assert len(baseline_cols) == 8
  assert len(enriched_cols) == 26
  assert len(llm_categorical_cols) == 4
  assert extraction_tool['name'] == 'extract_complaint_features'

def test_valid_repsonse_passes():
  _,_,_, extraction_tool = load_artifacts()

  good_response = {
       'harm_type': 'financial_loss',
    'severity': 'high',
    'discrimination_mentioned': False,
    'resolution_requested': 'refund',
    'sentiment_intensity': 'irate'
  }

  assert validate_llm_response(good_response, extraction_tool) is True

def test_invalid_enum_raises_error():
  _,_,_, extraction_tool = load_artifacts()

  bad_response = {
       'harm_type': 'made_up_value',
    'severity': 'high',
    'discrimination_mentioned': False,
    'resolution_requested': 'refund',
    'sentiment_intensity': 'irate'
  }

  with pytest.raises(ValueError):
      validate_llm_response(bad_response, extraction_tool)

def test_missing_field_error():
  _,_,_, extraction_tool = load_artifacts()

  incomplete_response = {'harm_type': 'financial_loss', 'severity': 'high'}

  with pytest.raises(ValueError):
    validate_llm_response(incomplete_response, extraction_tool)

Overwriting test_cfpb_logic.py


In [22]:
# test
!pip install pytest --quiet
!pytest test_cfpb_logic.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/cfpb_project
plugins: anyio-4.14.2, langsmith-0.12.1, typeguard-4.6.0
collected 4 items                                                              

test_cfpb_logic.py::test_load_artifacts_returns_correct_counts PASSED    [ 25%]
test_cfpb_logic.py::test_valid_repsonse_passes PASSED                    [ 50%]
test_cfpb_logic.py::test_invalid_enum_raises_error PASSED                [ 75%]
test_cfpb_logic.py::test_missing_field_error PASSED                      [100%]

============================== 4 passed in 0.81s ===============================


## Step 4 result

All 4 tests pass. Along the way, caught two real bugs: a duplicate variable
name (`enriched_cols` reused) that silently overwrote data during unpacking,
and a test function missing its `test_` prefix, so pytest wasn't collecting
it. Both fixed above.

## Step 5: Build the input row function

This function combines the structured Streamlit form inputs (product, issue, company, etc.) with the LLM-extracted features into a single dict representing one row of model input. This logic currently lives inline in app.py; pulling it into a standalone function makes it independently testable.

In [11]:
%%writefile cfpb_logic.py
import json
import pandas as pd

def load_artifacts():
  '''Load the core JSON artifacts used by the cfpb prediction pipeline'''
  with open('baseline_cols.json') as f:
    baseline_cols = json.load(f)
  with open('enriched_cols.json') as f:
    enriched_cols = json.load(f)
  with open('llm_categorical_cols.json') as f:
    llm_categorical_cols = json.load(f)
  with open('extraction_tool.json') as f:
    extraction_tool = json.load(f)
  return baseline_cols, enriched_cols, llm_categorical_cols, extraction_tool

def validate_llm_response(response:dict, extraction_tool:dict)-> bool:
  '''Check that an LLM response has a;; the required fields with allowed enum vlaues'''
  schema = extraction_tool['input_schema']['properties']
  required = extraction_tool['input_schema']['required']

  for field in required:
    if field not in response:
      raise ValueError(f"Missing rquired field {field}")

    if 'enum' in schema[field] and response[field] not in schema[field]['enum']:
      raise ValueError(f"Invalid value for {field}: {response[field]}")
  return True

def build_input_row(product: str, issue: str, sub_issue: str, company: str,
                    state: str, has_tag: bool, timely_response: bool,
                    llm_features: dict)-> dict:
    '''Combine structured from inputs and LLM features into one row dict.'''
    row = {
        'product_clean': product,
        'Issue': issue,
        'Sub-issue' : sub_issue,
        'Company': company,
        'State' : state,
        'has_tag': int(has_tag),
        'has_narrative': 1,
        'timely_response': int(timely_response)
    }
    row.update(llm_features)
    return row

Overwriting cfpb_logic.py


In [12]:
# Sanity check

import importlib
import cfpb_logic
importlib.reload(cfpb_logic)
from cfpb_logic import build_input_row

llm_features = {
    'harm_type': 'financial_loss',
    'severity': 'high',
    'discrimination_mentioned': False,
    'resolution_requested': 'refund',
    'sentiment_intensity': 'irate'
}

row = build_input_row('credit reporting', 'Incorrect information',
                      'Information belongs to someone else',
                   'Experian', 'NY', True, True, llm_features)
print(row)
print('Number of Keys:', len(row))

{'product_clean': 'credit reporting', 'Issue': 'Incorrect information', 'Sub-issue': 'Information belongs to someone else', 'Company': 'Experian', 'State': 'NY', 'has_tag': 1, 'has_narrative': 1, 'timely_response': 1, 'harm_type': 'financial_loss', 'severity': 'high', 'discrimination_mentioned': False, 'resolution_requested': 'refund', 'sentiment_intensity': 'irate'}
Number of Keys: 13


## Step 5b: Write pytest tests for build_input_row

We test that the function correctly combines structured inputs with LLM features, that boolean fields are properly converted to integers, and
that the final row contains the expected number of keys.

In [13]:
%%writefile test_cfpb_logic.py
import pytest
from cfpb_logic import load_artifacts, validate_llm_response, build_input_row

def test_load_artifacts_returns_correct_counts():
  baseline_cols, enriched_cols, llm_categorical_cols, extraction_tool = load_artifacts()
  assert len(baseline_cols) == 8
  assert len(enriched_cols) == 26
  assert len(llm_categorical_cols) == 4
  assert extraction_tool['name'] == 'extract_complaint_features'

def test_valid_repsonse_passes():
  _,_,_, extraction_tool = load_artifacts()

  good_response = {
       'harm_type': 'financial_loss',
    'severity': 'high',
    'discrimination_mentioned': False,
    'resolution_requested': 'refund',
    'sentiment_intensity': 'irate'
  }

  assert validate_llm_response(good_response, extraction_tool) is True

def test_invalid_enum_raises_error():
  _,_,_, extraction_tool = load_artifacts()

  bad_response = {
    'harm_type': 'made_up_value',
    'severity': 'high',
    'discrimination_mentioned': False,
    'resolution_requested': 'refund',
    'sentiment_intensity': 'irate'
  }

  with pytest.raises(ValueError):
      validate_llm_response(bad_response, extraction_tool)

def test_missing_field_error():
  _,_,_, extraction_tool = load_artifacts()

  incomplete_response = {'harm_type': 'financial_loss', 'severity': 'high'}

  with pytest.raises(ValueError):
    validate_llm_response(incomplete_response, extraction_tool)

def test_build_input_row_has_correct_key_count():
    llm_features = {
        'harm_type': 'financial_loss',
        'severity': 'high',
        'discrimination_mentioned': False,
        'resolution_requested': 'refund',
        'sentiment_intensity': 'irate'
    }
    row = build_input_row('Credit reporting','Incorrect information',
                          'Information belongs to someone else', 'Experian',
                          'NY', True, True, llm_features)
    assert len(row) == 13

def test_build_input_row_converts_boolean_to_int():
  llm_features = {
       'harm_type': 'financial_loss',
        'severity': 'high',
        'discrimination_mentioned': False,
        'resolution_requested': 'refund',
        'sentiment_intensity': 'irate'
  }
  row = build_input_row('Credit reporting', 'Issue', 'Sub-issue', 'Company',
                        'NY', True, False, llm_features)
  assert row['has_tag'] == 1
  assert row['timely_response'] == 0

Overwriting test_cfpb_logic.py


In [14]:
!pytest test_cfpb_logic.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/cfpb_project
plugins: anyio-4.14.2, typeguard-4.6.0, langsmith-0.12.1
collected 6 items                                                              

test_cfpb_logic.py::test_load_artifacts_returns_correct_counts PASSED    [ 16%]
test_cfpb_logic.py::test_valid_repsonse_passes PASSED                    [ 33%]
test_cfpb_logic.py::test_invalid_enum_raises_error PASSED                [ 50%]
test_cfpb_logic.py::test_missing_field_error PASSED                      [ 66%]
test_cfpb_logic.py::test_build_input_row_has_correct_key_count PASSED    [ 83%]
test_cfpb_logic.py::test_build_input_row_converts_boolean_to_int PASSED  [100%]

============================== 6 passed in 0.56s ===============================


## Step 5 result
`build_input_row` combines structured form inputs with LLM-extracted features into a single row dict (13 keys total), correctly converting boolean fields to integers. All 6 tests now pass, including a caught regression where a bug fixed earlier in `validate_llm_response` had crept back into the file during a rewrite.

## Step 6: CI/CD with GitHub Actions

`cfpb_logic.py`, `test_cfpb_logic.py`, and the four JSON artifacts were
pushed to the project's GitHub repo, along with a workflow file at
`.github/workflows/test.yml` that runs the full pytest suite automatically
on every push.

The first run failed with `FileNotFoundError`, since `load_artifacts()`
referenced the JSON files directly (`baseline_cols.json`) while they
actually live in an `artifacts/` subfolder in the repo. Updating the file
paths in `load_artifacts()` to `artifacts/baseline_cols.json` (and the
same for the other three files) resolved this, and the workflow now
passes: all 6 tests run automatically on every push, with no manual
setup required.

This closes the loop from "I wrote tests locally" to "these tests run
themselves on every change," which is the actual point of CI.